# 04 Results Analysis

Compare classical ML, TabNet, TabTransformer, TabM-style deep learning, and ensembles. This notebook is designed to support the final project report, including why TabNet may underperform on tabular data.


In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
from src.config import METRICS_PATH, FIGURES_DIR, RESULTS_DIR
from src.plots import plot_metric_comparison

metrics = pd.read_csv(METRICS_PATH)
metrics


## Clean Results Table


In [ ]:
cols = ['model', 'roc_auc', 'pr_auc', 'recall', 'f1', 'precision', 'threshold', 'threshold_source']
clean = metrics[metrics['split'] == 'test'][cols].sort_values('pr_auc', ascending=False)
clean


## XGBoost vs TabNet vs TabTransformer vs TabM


In [ ]:
key_models = metrics[metrics['model'].str.contains('XGBoost|TabNet|TabTransformer|TabM', regex=True, na=False)]
key_models[['model', 'roc_auc', 'pr_auc', 'recall', 'f1', 'threshold_source']].sort_values('pr_auc', ascending=False)


## Best Models By Metric


In [ ]:
for metric in ['roc_auc', 'pr_auc', 'recall', 'f1']:
    row = metrics.loc[metrics[metric].idxmax()]
    print(f'Best {metric}: {row.model} = {row[metric]:.4f}')


## Final Plots


In [ ]:
for metric in ['roc_auc', 'pr_auc', 'recall', 'f1']:
    plot_metric_comparison(metrics, metric, 'test', FIGURES_DIR / f'{metric}_comparison.png')
list(FIGURES_DIR.glob('*threshold_curve.png'))[:10]


## Fairness Analysis


In [ ]:
fairness_files = sorted(RESULTS_DIR.glob('fairness_*.csv'))
if fairness_files:
    fairness = pd.read_csv(fairness_files[-1])
    display(fairness.sort_values(['model', 'attribute', 'group']).head(40))
    fairness_summary = fairness.groupby(['model', 'attribute']).agg(
        min_recall=('recall', 'min'),
        max_recall=('recall', 'max'),
        mean_recall=('recall', 'mean'),
        min_pr_auc=('pr_auc', 'min'),
        max_pr_auc=('pr_auc', 'max')
    ).reset_index()
    display(fairness_summary)
else:
    print('No fairness CSV found. Run src.run_experiment first.')


## Interpretation: Why XGBoost Is Still Strong

Gradient boosting remains hard to beat on medium-sized heterogeneous tabular data because trees naturally handle nonlinear thresholds, sparse interactions, mixed feature types after encoding, and irregular missingness patterns. XGBoost also optimizes robustly with relatively little architecture tuning, while neural tabular models are more sensitive to representation, regularization, learning rate, and class imbalance.


## Interpretation: Why TabNet Is Not Optimal Here

TabNet uses sequential feature selection through attentive masks. That can be interpretable, but it may miss global feature interactions that boosted trees capture through many additive splits. It can also be harder to optimize on noisy, imbalanced clinical tabular data. Unlike images or language, this dataset has no spatial or temporal structure for deep networks to exploit directly, so stronger inductive bias from boosting can dominate.


## Interpretation: TabTransformer and TabM

TabTransformer models categorical features with self-attention, which is more global than TabNet masks, while the TabM-style model approximates an ensemble by averaging multiple prediction heads over a shared representation. These models test whether newer tabular DL ideas close the gap with XGBoost. Success should be judged by PR-AUC, recall at clinically meaningful thresholds, and stability under patient-level splitting rather than only by raw accuracy.


## Interpretation: Ensembles

The runner now evaluates XGBoost + TabNet, XGBoost + TabTransformer, XGBoost + TabM, and logistic-regression stacking over model probabilities. If a weighted ensemble selects almost all XGBoost weight, that is evidence that the deep model is not adding complementary validation signal. If stacking improves PR-AUC or recall/F1, include that as the strongest ensemble result.


## Limitations

Important limitations to discuss: encounter-level splits can overestimate generalization when patients repeat; patient-level splits are stricter. Subgroup fairness metrics by age and gender are descriptive and may be noisy in small groups. TabPFN and TabR are not default models because their official workflows require additional dependencies, constraints, or retrieval infrastructure beyond the core Colab-friendly implementation.
